In [5]:
import pandas as pd

path_test = '../data/raw/hi/test_split.parquet'
csv_test = '../data/processed/hi_test_437.csv'

path_train = '../data/raw/hi/train_split.parquet'
csv_train = '../data/processed/hi_train_3489.csv'


csv_val = '../data/processed/hi_val_437.csv'

try:
    #convert testset to csv
    df_test = pd.read_parquet(path_test, engine='pyarrow')
    test_size = len(df_test)

    print(f"Hinglish testset loaded. Total rows: {len(df_test)}")
    print("Columns found:", df_test.columns.tolist())


    df_test.to_csv(csv_test, index=False, encoding='utf-8')
    print(f"Testset saved as '{csv_test}'")



    #divide trainset into train and validation set, convert to csv
    df = pd.read_parquet(path_train, engine='pyarrow')

    df_val = df.iloc[:test_size].copy()
    df_train = df.iloc[test_size:].copy()

    print(f"Hinglish training set loaded. Total rows: {len(df_train)}")
    print(f"Hinglish validation set loaded. Total rows: {len(df_val)}")
    print("Columns found:", df.columns.tolist())


    df_train.to_csv(csv_train, index=False, encoding='utf-8')
    print(f"Training set saved as '{csv_test}'")

    df_val.to_csv(csv_val, index=False, encoding='utf-8')
    print(f"Validation set saved as '{csv_val}'")

except Exception as e:
    print(f"Error occurred: {e}")

Hinglish testset loaded. Total rows: 437
Columns found: ['text', 'toxic']
Testset saved as '../data/processed/hi_test_437.csv'
Hinglish training set loaded. Total rows: 3489
Hinglish validation set loaded. Total rows: 437
Columns found: ['text', 'toxic']
Training set saved as '../data/processed/hi_test_437.csv'
Validation set saved as '../data/processed/hi_val_437.csv'


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split


df_en_full = pd.read_csv('../data/raw/en/textdetox_en.csv')
df_test_multi = pd.read_csv('../data/processed/textdetox_multilingual_en_ru_hi_sample_504.csv')

print(f'full size {len(df_en_full)}')
print(f'test_multi size {len(df_test_multi)}')


df_en_full['clean_text'] = df_en_full['text'].astype(str).str.strip()
df_test_multi['clean_text'] = df_test_multi['text'].astype(str).str.strip()


overlap_count = df_en_full['clean_text'].isin(df_test_multi['clean_text']).sum()
print(f"Found {overlap_count} overlapping samples between Train and Test sets.")


df_filtered = df_en_full[~df_en_full['clean_text'].isin(df_test_multi['clean_text'])].copy()
df_filtered = df_filtered.drop(columns=['clean_text'])


df_train, df_val = train_test_split(
    df_filtered, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_filtered['toxic']
)

print(f"English train size: {len(df_train)} samples")
print(f"English val size: {len(df_val)} samples")


df_train.to_csv('../data/processed/en_train_3860.csv', index=False, encoding='utf-8')
df_val.to_csv('../data/processed/en_val_966.csv', index=False, encoding='utf-8')


full size 5000
test_multi size 504
Found 174 overlapping samples between Train and Test sets.
English train size: 3860 samples
English val size: 966 samples


In [7]:
import pandas as pd


en_train = '../data/processed/en_train_3860.csv'
hi_train = '../data/processed/hi_train_3489.csv'
ru_train = '../data/raw/ru/ru_trainset.csv'

en_val = '../data/processed/en_val_966.csv'
hi_val = '../data/processed/hi_val_437.csv'
ru_val = '../data/raw/ru/ru_valset.csv'


try:
    df1 = pd.read_csv(en_train)
    df2 = pd.read_csv(hi_train)
    df3 = pd.read_csv(ru_train)

    df4 = pd.read_csv(en_val)
    df5 = pd.read_csv(hi_val)
    df6 = pd.read_csv(ru_val)

    df3 = df3.sample(n=5000, random_state=42)
    df6 = df6.sample(n=1000, random_state=42)

    print(f"Loaded: train en ({len(df1)} rows), train hi ({len(df2)} rows), train ru ({len(df3)} rows)")

    print(f"Loaded: val en ({len(df4)} rows), val hi ({len(df5)} rows), val ru ({len(df6)} rows)")

    if 'toxic' in df1.columns:
        df1 = df1.rename(columns={'toxic': 'label'})
    if 'toxic' in df2.columns:
        df2 = df2.rename(columns={'toxic': 'label'})
    if 'toxic' in df4.columns:
        df4 = df4.rename(columns={'toxic': 'label'})
    if 'toxic' in df5.columns:
        df5 = df5.rename(columns={'toxic': 'label'})


    df_train_merged = pd.concat([df1, df2, df3], ignore_index=True)
    print(f"Train rows: {len(df_train_merged)}")

    df_val_merged = pd.concat([df4, df5, df6], ignore_index=True)
    print(f"Val rows: {len(df_val_merged)}")

    df_train_shuffled = df_train_merged.sample(frac=1, random_state=42).reset_index(drop=True)
    df_val_shuffled = df_val_merged.sample(frac=1, random_state=42).reset_index(drop=True)


    output_train = '../data/processed/train_en_hi_ru_12349.csv'
    output_val = '../data/processed/val_en_hi_ru_2403.csv'
    df_train_shuffled.to_csv(output_train, index=False, encoding='utf-8')
    df_val_shuffled.to_csv(output_val, index=False, encoding='utf-8')

    print(f"Combined and shuffled train dataset saved as '{output_train}' ({len(df_train_shuffled)} rows)")
    print(f"Combined and shuffled val dataset saved as '{output_val}' ({len(df_val_shuffled)} rows)")

except FileNotFoundError as e:
    print(f"Error: One of the files was not found! Details: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded: train en (3860 rows), train hi (3489 rows), train ru (5000 rows)
Loaded: val en (966 rows), val hi (437 rows), val ru (1000 rows)
Train rows: 12349
Val rows: 2403
Combined and shuffled train dataset saved as '../data/processed/train_en_hi_ru_12349.csv' (12349 rows)
Combined and shuffled val dataset saved as '../data/processed/val_en_hi_ru_2403.csv' (2403 rows)
